In [1]:
import torch
import torch.nn as nn
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler, StandardScaler
import numpy as np
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
class AE(nn.Module):
    def __init__(self, input_size, latent_size):
        super(AE, self).__init__()
        self.input_size = input_size
        self.latent_size = latent_size

        # encoder network
        self.encoder = nn.Sequential(
            nn.Linear(input_size, 56), nn.ReLU(),
            nn.Linear(56, 28), nn.ReLU(),
            nn.Linear(28, latent_size)
        )
        # decoder network
        self.decoder = nn.Sequential(
            nn.Linear(latent_size, 28), nn.ReLU(),
            nn.Linear(28, 56), nn.ReLU(),
            nn.Linear(56, input_size)
        )

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        z = self.encode(x)
        return self.decode(z)
        
#load trained model
def load_model(input_size, latent_size):
    model = AE(input_size=input_size, latent_size=latent_size)
    model.load_state_dict(
        torch.load("/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/Autoencoder/best_autoencoder_nested_cv.pt")
    )
    model.eval()
    return model
model = load_model(103,10)

In [4]:
#now only transform the unseen data to see how well the PCA encodes for the different variables
df_ae_unseen  = pd.read_csv("/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/Data/A2_unseen_data_ukb_adni_o3.csv")
df_ae_unseen = df_ae_unseen.iloc[:,1:]
df_ae_unseen = df_ae_unseen.iloc[:,5:]

#unseen data 
scaler = RobustScaler()
data_scaled = scaler.fit_transform(df_ae_unseen)
data_tensor = torch.tensor(data_scaled, dtype=torch.float32)
tensor_dataset = TensorDataset(data_tensor)

def evaluate(model, dataloader):
    model.eval()
    all_outputs = []
    all_inputs = []

    with torch.no_grad():
        for x, in dataloader:
            output = model(x)
            all_outputs.append(output.numpy())
            all_inputs.append(x.numpy())

    all_outputs = np.concatenate(all_outputs, axis=0)
    all_inputs = np.concatenate(all_inputs, axis=0)

    mse = mean_squared_error(all_inputs, all_outputs)
    mae = mean_absolute_error(all_inputs, all_outputs)
    r2 = r2_score(all_inputs, all_outputs)

    return mse, mae, r2

test_dataloader = DataLoader(tensor_dataset, shuffle=False)
mse, mae, r2 = evaluate(model, test_dataloader)

In [5]:
mse, mae, r2

(0.153629869222641, 0.2890178859233856, 0.7262091040611267)